### Test AI Agent 🤖 Tool Calling with DeepEval 🧪

Testing AI Agent involves testing of the Tools being invoked by an AI Agent. Here, AI Agent will invoke the necessary tools based on the given input and respond with the help of the tools being bounded with the AI Agent


<img src="./img/AIAGent.png" width="800" height="400" style="display: block; margin: auto;">

In [6]:
# !pip install -qU duckduckgo-search
%pip install -U ddgs


   ----------- ---------------------------- 2/7 [hyperframe]
   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------------------- ----- 6/7 [ddgs]
   ---------------------------------- ----- 6/7 [ddgs]
   ---------------------------------------- 7/7 [ddgs]

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

#### AI Agent with Tools

In [ ]:
from langchain.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

@tool
def add_numbers(a: int, b: int) -> int:
    "Add two numbers and return results."
    return int(a) + int(b)

@tool
def subtract_numbers(a: int, b: int) -> int:
    "Subtract two numbers and return results."
    return int(a) - int(b)

tools = [add_numbers, subtract_numbers, search_tool]

agent = initialize_agent(
    tools= tools,
    llm= llm,
    agent= AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose= True,
    return_intermediate_steps= True # keeps track of which tool was used and how.
)

response = agent.invoke("Who is the current president of USA in 2025, just give the name")

print(response)

def query_ai_agent(question):
    response = agent.invoke(question)
    intermediate_steps = response['intermediate_steps']
    agent_action, results = intermediate_steps[0]
    tool = agent_action.tool
    tool_input = agent_action.tool_input
    return response, tool, tool_input


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7664\579676457.py:19: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Action:
```
{
  "action": "duckduckgo_search",
  "action_input": "current president of USA in 2025"
}
```
Observation: Donald Trump 's second and current tenure as the president of the United States began upon his inauguration as the 47th president on January 20, 2025 . Doug Burgum has served as Secretary of the Interior since February 2025 and previously served as governor of North Dakota from 2016 to 2024. The incumbent president , Joe Biden of the Democratic Party, initially ran for re-election as its presumptive nominee , 4 facing little opposition and ... Trump is President of the United States and is a Republican. He has served since Jan. 20, 2025 . Trump’s current term ends on Jan. 20, 2029. He is 79 years old. He was previously President of the United States as a Republican from 2017 to Jan. 20, 2021. Apr 15, 2025 · As of 2025 , the President of the United States is Donald J. Trump, a real estate mogul, television personality, and politic

- DuckDuckGoSearchRun is a tool wrapper (commonly used in frameworks like LangChain) that lets your code send a search query to DuckDuckGo and return the results in a usable format.
    - It takes a string query (e.g., "latest AI news")
    - It runs that query against DuckDuckGo’s search engine
    - It returns the search results (usually as text, sometimes summarized or parsed depending on the wrapper implementation)
    - Letting an AI agent fetch fresh information from the web.
    - Automating fact-checking or context retrieval.

In [1]:
response,tool, tool_input = query_ai_agent("Who is the president of USA in 2025, just give me the name")

print(response)

print(tool)

print(tool_input)

NameError: name 'query_ai_agent' is not defined

### Testing AI Agent with DeepEval

In [3]:
import deepeval

deepeval.login("confident_us_8k9P7QpyyKgjpa7yzXG0ULlki3JAwq0DPAstgNKA1x0=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [4]:
!deepeval set-ollama qwen2.5:latest

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen2.5:latest` for 
all evals that require an LLM.


In [5]:
from deepeval.test_case import ToolCall

test_data = [
    {
        "input": "What is the sum of 20 and 40",
        "expected_output": "60",
        "tool_called": [
            ToolCall(name = "add_numbers")
        ]
    },
     {
        "input": "Who is the president of USA in 2025, just give me the name",
        "expected_output": "Donald Trump",
        "tool_called": [
            ToolCall(name = "duckduckgo_search")
        ]
    }
]

In [6]:
test_data

[{'input': 'What is the sum of 20 and 40',
  'expected_output': '60',
  'tool_called': [ToolCall(
       name="add_numbers"
   )]},
 {'input': 'Who is the president of USA in 2025, just give me the name',
  'expected_output': 'Donald Trump',
  'tool_called': [ToolCall(
       name="duckduckgo_search"
   )]}]

In [7]:
test_data[0]['tool_called']

[ToolCall(
     name="add_numbers"
 )]

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ToolCorrectnessMetric

test_cases = []
for testcase in test_data:
  response,tool, tool_input = query_ai_agent(testcase['input'])
  test_case = LLMTestCase(
    input=testcase['input'],
    tools_called=[ToolCall(name=tool)],
    actual_output=response,
    expected_tools=testcase['tool_called']
  )

  test_cases.append(test_case)

NameError: name 'query_ai_agent' is not defined

In [86]:
test_cases

[LLMTestCase(input='What is the sum of 20 and 40', actual_output={'input': 'What is the sum of 20 and 40', 'output': 'The sum of 20 and 40 is 60.', 'intermediate_steps': [(AgentAction(tool='add_numbers', tool_input={'a': 20, 'b': 40}, log='Action:\n```\n{\n  "action": "add_numbers",\n  "action_input": {\n    "a": 20,\n    "b": 40\n  }\n}\n```'), 60)]}, expected_output=None, context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=[ToolCall(
     name="add_numbers"
 )], expected_tools=[ToolCall(
     name="add_numbers"
 )], reasoning=None, name=None),
 LLMTestCase(input='Who is the president of USA in 2025, just give me the name', actual_output={'input': 'Who is the president of USA in 2025, just give me the name', 'output': 'Donald Trump', 'intermediate_steps': [(AgentAction(tool='duckduckgo_search', tool_input='who is the president of USA in 2025', log='Thought: I can use DuckDuckGo to search for this information.\n\nAction:\n```\n{\n  "action": "duc

In [88]:
metrics = ToolCorrectnessMetric()

for testcase in test_cases:
    metrics.measure(test_case=testcase)
    print(metrics.score)
    print(metrics.reason)
    print(metrics.expected_tools)

/Users/karthik/tryout/aiqaDemo/myenv312/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0
All expected tools ['add_numbers'] were called (order not considered).
[ToolCall(
    name="add_numbers"
)]


1.0
All expected tools ['duckduckgo_search'] were called (order not considered).
[ToolCall(
    name="duckduckgo_search"
)]
